In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor
from sklearn.cross_decomposition import PLSRegression

import domain_adaptation as da

In [ ]:
def build_forecast_models(random_state=42):
    """
    Прогностические модели M_A / M_D и для сценариев без адаптации
    """
    return {
        'Linear': Pipeline([("scaler", StandardScaler()), ("lin", LinearRegression())]),
        'Ridge':  Pipeline([("scaler", StandardScaler()),
                            ("ridge", Ridge(alpha=1.5, random_state=random_state))]),
        'Lasso':  Pipeline([("scaler", StandardScaler()),
                            ("lasso", Lasso(alpha=5e-4, max_iter=500, tol=0.01,
                                            random_state=random_state))]),
        'LGBM':   Pipeline([("boost", LGBMRegressor(
                                n_estimators=1160, num_leaves=93, max_depth=5,
                                learning_rate=0.0144, subsample=0.56,
                                colsample_bytree=0.99, reg_alpha=0.023,
                                reg_lambda=0.006, min_child_samples=25,
                                random_state=random_state, verbose=-1))]),
        'MLP':    Pipeline([("scaler", StandardScaler()),
                            ("mlp", MLPRegressor(hidden_layer_sizes=(12,),
                                                 alpha=1e-4, batch_size=32,
                                                 learning_rate_init=1e-3,
                                                 early_stopping=True, max_iter=1500,
                                                 random_state=random_state))]),
    }

In [ ]:
def get_adaptation_model(name, random_state=42):
    """
    Модели для адаптации
    """
    if name == 'linreg':
        return LinearRegression(), False
    if name == 'boost':
        return LGBMRegressor(n_estimators=1000, reg_alpha=0.7, reg_lambda=0.7,
                             max_depth=3, subsample=0.8, colsample_bytree=0.8,
                             random_state=random_state, verbose=-1), True  # MultiOutput
    if name == 'pls':
        return PLSRegression(n_components=100, scale=True, max_iter=2000), False
    raise ValueError(name)
def field_model_factory():
    return Pipeline([("scaler", StandardScaler()), ("lin", LinearRegression())])
def wind_model_factory_lgbm():
    return Pipeline([("scaler", StandardScaler()),
                     ("boost", LGBMRegressor(n_estimators=1000, reg_alpha=0.7,
                                             reg_lambda=0.7, max_depth=3,
                                             subsample=0.8, colsample_bytree=0.8,
                                             random_state=42, verbose=-1))])

In [ ]:
# data_ace_24 = pd.read_csv("../data/Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
# data_discover_24 = pd.read_csv("../data/Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

data_ace_af = pd.read_csv("../data/Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("../data/Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [ ]:
# 0. Копирование загруженных данных в отдельные переменные

# data_ace = data_ace_24.copy()
# data_disc = data_discover_24.copy()

data_ace = data_ace_af.copy()
data_disc = data_discover_af.copy()

In [ ]:
# 1. Уменьшение датасет до нужного промежутка времени

# cutoff = pd.Timestamp('2023-12-31 23:59:59')

# data_ace = data_ace[data_ace.index <= cutoff]
# data_disc = data_disc[data_disc.index <= cutoff]

In [ ]:
# 2. Задание переменных для обучения

split_date = "2022-01-01"
# delays = "24h"
delays = "auto_func"
n_seeds = 2
future_lags = [f'Dst_plus{i}' for i in range(1, 25)]
# targets = ['Dst_plus1', 'Dst_plus2', 'Dst_plus3', 'Dst_plus6', 'Dst_plus12', 'Dst_plus24']
targets = ['Dst_plus1', 'Dst_plus3', 'Dst_plus6']
results_list = []

In [ ]:
# 3. Сценарии без адаптации

for target in targets:
    for scen in ['ace-ace', 'ace-disc', 'disc-ace', 'disc-disc']:
        print(f"\n=== {scen.upper()} | {target} ===")
        da.run_no_adaptation_scenario(
            data_ace, data_disc, split_date, target, future_lags,
            scenario=scen, build_models_fn=build_forecast_models,
            delays=delays, n_seeds=n_seeds, results_list=results_list,
        )
    print(f"\n=== COMBINED | {target} ===")
    da.run_combined_scenario(
        data_ace, data_disc, split_date, target, future_lags,
        build_models_fn=build_forecast_models,
        delays=delays, n_seeds=n_seeds, results_list=results_list,
    )

In [ ]:
# 4. ДА DSCOVR -> ACE

for adapt_name in ['linreg', 'boost', 'pls']:
    model, multi = get_adaptation_model(adapt_name)
    disc_test_adapted, _ = da.adapt_dscovr_to_ace(
        data_ace, data_disc, future_lags, split_date,
        adapt_model=model, multi_output=multi,
    )
    for target in targets:
        print(f"\n=== ДА D-A {adapt_name} | {target} ===")
        da.run_adaptation_D2A(
            data_ace, disc_test_adapted, split_date, target, future_lags,
            build_models_fn=build_forecast_models,
            adaptation_label=f'da-d-a-{adapt_name}',
            delays=delays, n_seeds=n_seeds, results_list=results_list,
        )

In [ ]:
# 5. ДА ACE -> DSCOVR

for adapt_name in ['linreg', 'boost', 'pls']:
    model, multi = get_adaptation_model(adapt_name)
    ace_old_adapted, disc_train, disc_test, _ = da.adapt_ace_to_dscovr(
        data_ace, data_disc, future_lags, split_date,
        adapt_model=model, multi_output=multi,
    )
    for target in targets:
        print(f"\n=== ДА A-D {adapt_name} | {target} ===")
        da.run_adaptation_A2D(
            ace_old_adapted, disc_train, disc_test, target, future_lags,
            build_models_fn=build_forecast_models,
            adaptation_label=f'da-a-d-{adapt_name}',
            delays=delays, n_seeds=n_seeds, results_list=results_list,
        )

In [ ]:
# 6. Cross-domain адаптация (ЛР+ГБ)

# DSCOVR -> ACE
disc_test_cross = da.cross_adapt_domain(
    src_df=data_disc, tgt_df=data_ace, future_lags=future_lags,
    field_model_factory=field_model_factory,
    wind_model_factory=wind_model_factory_lgbm,
    direction='disc-to-ace', split_date=split_date,
)
for target in targets:
    da.run_adaptation_D2A(
        data_ace, disc_test_cross, split_date, target, future_lags,
        build_models_fn=build_forecast_models,
        adaptation_label='da-d-a-cross-lr+lgbm',
        n_seeds=n_seeds, results_list=results_list,
    )


In [ ]:
df_results = pd.DataFrame(results_list)
df_results

In [ ]:
df_results.to_csv("results_all.csv", index=False, sep=';', decimal=',')